In [1]:
!pip install langchain langchain_core langchain_openai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.6/987.6 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.7/371.7 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.5/328.5 kB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.9 MB/s eta 0:00:00


In [2]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass.getpass()

··········


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain.memory import ChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory

In [4]:
employee_performance_data = """date,employee_id,rating
2024-01-01,1,4.5
2024-01-01,2,3.8
2024-01-02,1,4.6
2024-01-02,2,3.9
2024-01-03,1,4.7
2024-01-03,2,4.0
2024-01-04,1,4.8
2024-01-04,2,4.1
2024-01-05,1,2.0
2024-01-05,2,4.2
2024-01-06,1,4.6
2024-01-06,2,4.3
2024-01-07,1,4.7
2024-01-07,2,4.4
2024-01-08,1,4.8
2024-01-08,2,4.5
2024-01-09,1,4.7
2024-01-09,2,2.0
2024-01-10,1,4.6
2024-01-10,2,4.3"""

In [5]:
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

In [6]:
prompt = ChatPromptTemplate.from_messages(
    [("system", """You are a data analyst for Wired Brain Coffee, a coffee house chain with over 1600 locations worldwide.
      You will be given data in the form of a CSV. Your job is to determine if any anomalies are present the data."""),
     MessagesPlaceholder(variable_name="chat_history"),
     ("user", "{input}")])

chain = prompt | llm

chat_history = ChatMessageHistory()

anomaly_chain = RunnableWithMessageHistory(chain,
                                           get_session_history=lambda session_id: chat_history,
                                           input_messages_key="input",
                                           history_messages_key="chat_history")

In [7]:
input_message = """Below is the Below is the data to analyze. Are there any anomalies present?
 Make sure to also ignore the first line since these are the column titles:
    \n\n{0}""".format(employee_performance_data)
response = anomaly_chain.invoke({"input": input_message}, {"configurable": {"session_id": "unused"}})

In [8]:
print(response.content)

Based on the data provided, there are two anomalies in the ratings:

1. On 2024-01-05, employee 1 has a rating of 2.0, which is significantly lower than their other ratings, which consistently range from 4.5 to 4.8.
2. On 2024-01-09, employee 2 has a rating of 2.0, which is also significantly lower compared to their other ratings, which consistently range from 3.8 to 4.5.

These sudden drops in ratings for both employees on specific days are unusual compared to the trend observed on other days and could indicate an issue or error that might need further investigation.


In [9]:
input_message = "Are you sure? Double-check your findings and tell me the anomalies when you are more sure."
response = anomaly_chain.invoke({"input": input_message}, {"configurable": {"session_id": "unused"}})

In [10]:
print(response.content)

Upon re-examining the data provided, I can confirm the anomalies previously identified:

1. **For Employee 1:**
   - On **2024-01-05**, the rating is **2.0**. This is significantly lower than the usual ratings for this employee, which generally range from 4.5 to 4.8.

2. **For Employee 2:**
   - On **2024-01-09**, the rating is **2.0**. This is a sharp drop compared to the typical ratings for this employee, which are usually between 3.8 and 4.5.

These anomalies stand out as they deviate markedly from the pattern of ratings on other days for each respective employee. This could suggest a potential error in data entry or an actual issue on those specific days that affected the ratings dramatically. Further investigation into the circumstances or a review of data entry practices on those dates might be warranted.
